In that notebook, I'll try to solve the Titanic surviving predition problem using PyTorch. I've already done some EDA before trying classical ML methods in https://www.kaggle.com/code/fourteenflames/titanic, so I don't repeat these steps here, but go on to preprocessing.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
df_train = pd.read_csv("/kaggle/input/titanic/train.csv", index_col = 'PassengerId')

In [3]:
df_test = pd.read_csv("/kaggle/input/titanic/test.csv", index_col = 'PassengerId')

In [4]:
df_train['Title'] = df_train['Name'].str.extract('([A-Za-z]+)\.')
df_train

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title
PassengerId,,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr
...,...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,Rev
888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,Miss
889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S,Miss


In [5]:
df_test['Title'] = df_test['Name'].str.extract('([A-Za-z]+)\.')
df_test

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title
PassengerId,,,,,,,,,,,
892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,Mr
893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,Mrs
894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,Mr
895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,Mr
896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,Mrs
...,...,...,...,...,...,...,...,...,...,...,...
1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S,Mr
1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,Dona
1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,Mr


In [6]:
# idea from CORAZON17 
def convert_title(title):
    if title in ["Ms", "Mlle", "Miss"]:
        return "Miss"
    elif title in ["Mme", "Mrs", "Countess", "Lady", "Dona"]:
        return "Mrs"
    elif title in ["Mr", "Major", "Col", "Capt", "Sir", "Don", "Jonkheer"]:
        return "Mr"
    elif title == "Master":
        return "Master"
    elif title == "Rev":
        return "Rev"
    else:
        return title   

In [7]:
y_train = df_train.Survived
X_train_full = df_train.drop(['Survived'], axis=1)
X_train = X_train_full.drop(['Name','Ticket', 'Cabin'], axis=1)
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
PassengerId,,,,,,,,
1,3,male,22.0,1,0,7.2500,S,Mr
2,1,female,38.0,1,0,71.2833,C,Mrs
3,3,female,26.0,0,0,7.9250,S,Miss
4,1,female,35.0,1,0,53.1000,S,Mrs
5,3,male,35.0,0,0,8.0500,S,Mr
...,...,...,...,...,...,...,...,...
887,2,male,27.0,0,0,13.0000,S,Rev
888,1,female,19.0,0,0,30.0000,S,Miss
889,3,female,NaN,1,2,23.4500,S,Miss


In [8]:
X_test = df_test.drop(['Name','Ticket', 'Cabin'], axis=1)
X_test

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
PassengerId,,,,,,,,
892,3,male,34.5,0,0,7.8292,Q,Mr
893,3,female,47.0,1,0,7.0000,S,Mrs
894,2,male,62.0,0,0,9.6875,Q,Mr
895,3,male,27.0,0,0,8.6625,S,Mr
896,3,female,22.0,1,1,12.2875,S,Mrs
...,...,...,...,...,...,...,...,...
1305,3,male,NaN,0,0,8.0500,S,Mr
1306,1,female,39.0,0,0,108.9000,C,Dona
1307,3,male,38.5,0,0,7.2500,S,Mr


In [9]:
X_train["Title"] = df_train["Title"].map(convert_title)
X_test["Title"] = df_test["Title"].map(convert_title)

In [10]:
for title in X_train["Title"].value_counts().keys():
    m = X_train[X_train["Title"]==title]['Age'].median()
    print(title, m)
    X_train.loc[(X_train.Age.isnull()) & (X_train["Title"]==title), 'Age'] = m
    X_test.loc[(X_test.Age.isnull()) & (X_test["Title"]==title), 'Age'] = m

Mr 30.0
Miss 21.0
Mrs 35.0
Master 3.5
Dr 46.5
Rev 46.5


In [11]:
X_train['Embarked'].fillna(value=X_train['Embarked'].mode()[0],inplace=True)


In [12]:
X_train = pd.get_dummies(X_train)
X_train

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Dr,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rev
PassengerId,,,,,,,,,,,,,,,,
1,3,22.0,1,0,7.2500,0,1,0,0,1,0,0,0,1,0,0
2,1,38.0,1,0,71.2833,1,0,1,0,0,0,0,0,0,1,0
3,3,26.0,0,0,7.9250,1,0,0,0,1,0,0,1,0,0,0
4,1,35.0,1,0,53.1000,1,0,0,0,1,0,0,0,0,1,0
5,3,35.0,0,0,8.0500,0,1,0,0,1,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,2,27.0,0,0,13.0000,0,1,0,0,1,0,0,0,0,0,1
888,1,19.0,0,0,30.0000,1,0,0,0,1,0,0,1,0,0,0
889,3,21.0,1,2,23.4500,1,0,0,0,1,0,0,1,0,0,0


In [13]:
X_test = pd.get_dummies(X_test)
X_test

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Dr,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rev
PassengerId,,,,,,,,,,,,,,,,
892,3,34.5,0,0,7.8292,0,1,0,1,0,0,0,0,1,0,0
893,3,47.0,1,0,7.0000,1,0,0,0,1,0,0,0,0,1,0
894,2,62.0,0,0,9.6875,0,1,0,1,0,0,0,0,1,0,0
895,3,27.0,0,0,8.6625,0,1,0,0,1,0,0,0,1,0,0
896,3,22.0,1,1,12.2875,1,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1305,3,30.0,0,0,8.0500,0,1,0,0,1,0,0,0,1,0,0
1306,1,39.0,0,0,108.9000,1,0,1,0,0,0,0,0,0,1,0
1307,3,38.5,0,0,7.2500,0,1,0,0,1,0,0,0,1,0,0


In [14]:
X_train = X_train.to_numpy().astype('float32')
y_train = y_train.to_numpy().astype('float32')
print(X_train.shape)
print(X_train.ndim)
print(y_train.shape)
print(y_train.ndim)

(891, 16)
2
(891,)
1


In [15]:
y_train = y_train.reshape(len(y_train), 1)
print(y_train.shape)
print(y_train.ndim)

(891, 1)
2


In [16]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

In [17]:
# by ELIAS 

class titanicDataset(Dataset):
    def __init__(self,x,y):
        self.X = torch.from_numpy(x)
        self.Y = torch.from_numpy((y))
        
    def __len__(self):
        return len(self.X)

    def __getitem__(self, item):
        return self.X[item], self.Y[item]

In [18]:
train_dataset = titanicDataset(X_train,y_train)
train_dl = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [19]:
class BinaryModel(torch.nn.Module):
    def __init__(self, n_inputs):
        super(BinaryModel, self).__init__()
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(n_inputs, 200),            
            torch.nn.ReLU(),
            torch.nn.Linear(200, 300),            
            torch.nn.ReLU(),
            torch.nn.Linear(300, 200),
            torch.nn.ReLU(),
            torch.nn.Linear(200, 1),
            torch.nn.Sigmoid()
         )

    def forward(self, X):
        return self.layers(X)        

In [20]:
model = BinaryModel(X_train.shape[1])

In [21]:
loss_fn = torch.nn.modules.loss.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
# optimizer = torch.optim.Adam(model.parameters(), lr=0.00001)
for epoch in range(1000):
    for X, Y in train_dl:
        optimizer.zero_grad()
        Y_hat = model(X)
        loss = loss_fn(Y_hat, Y)
        loss.backward()
        optimizer.step()

In [22]:
X_test = X_test.to_numpy().astype('float32')
print(X_test.shape)
print(X_test.ndim)


(418, 16)
2


In [23]:
with torch.no_grad():
    y_predict = model(torch.from_numpy(X_test))

In [24]:
y_predict = y_predict.numpy()
y_predict = y_predict[:,0]
y_predict = np.where(y_predict > 0.5, 1, 0)

In [25]:
output = pd.DataFrame({'PassengerId': df_test.index, 'Survived': y_predict})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
